# 🏥 **MediAI: Healthcare Diagnosis Intelligence Agent**
### *by Aigenthix*

---

A complete AI-powered Healthcare Diagnosis Agent built with:
- **LangGraph** — 13-node agent workflow
- **Groq API** — llama-3.1-8b-instant LLM
- **Gradio** — Professional healthcare UI (matching React design)
- **OpenCV / Pillow** — Medical image preprocessing

---

## 📦 Step 1: Install Dependencies

In [1]:
!pip install -q groq langgraph langchain-core gradio Pillow opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.0 MB/s eta 0:00:00


## 🔑 Step 2: Configure Groq API Key

In [2]:
import os
from getpass import getpass

GROQ_API_KEY = getpass("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")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ API Key configured.")

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··········
✅ API Key configured.


## 🧠 Step 3: Constants, System Prompts & Utilities

In [3]:
import re
import json
import time
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import cv2
from datetime import datetime
from groq import Groq
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional, Any

# ══════════════════════════════════════════════════════════════
# CONSTANTS — exact match to React UI
# ══════════════════════════════════════════════════════════════

SPECIALTIES = ["General Medicine", "Radiology", "Cardiology", "Dermatology",
    "Ophthalmology", "Neurology", "Orthopedics", "Pulmonology",
    "Gastroenterology", "Nephrology", "Endocrinology", "Oncology"]

DIAGNOSIS_TYPES = ["Preliminary Screening", "Differential Diagnosis",
    "Risk Assessment", "Follow-up Evaluation", "Emergency Triage"]

IMAGE_TYPES = ["X-Ray", "CT Scan", "MRI", "Ultrasound", "Skin Photo",
    "Eye Scan", "Pathology Slide", "ECG", "Blood Report", "Urine Report", "Other"]

CHRONIC_CONDITIONS = [
    "\U0001fa78 Diabetes (Type 1/2)", "\u2764\ufe0f Hypertension",
    "\U0001fac0 Heart Disease", "\U0001fad8 Kidney Disease",
    "\U0001f98b Thyroid Disorders", "\U0001fac1 Asthma / COPD",
    "\U0001f7e4 Liver Disease", "\U0001f9b4 Arthritis"
]

LANGUAGES = ["English", "Hindi", "Spanish", "French", "German",
    "Mandarin", "Arabic", "Portuguese", "Kannada", "Tamil",
    "Telugu", "Bengali", "Urdu", "Japanese", "Korean"]

WORKFLOW_NODES = [
    {"id": 1, "label": "Patient Input", "icon": "\U0001f4e5", "color": "#0EA5E9"},
    {"id": 2, "label": "Preprocessing", "icon": "\u2699\ufe0f", "color": "#8B5CF6"},
    {"id": 3, "label": "Image Analysis", "icon": "\U0001f52c", "color": "#EC4899"},
    {"id": 4, "label": "Feature Extraction", "icon": "\U0001f9ec", "color": "#F59E0B"},
    {"id": 5, "label": "Symptom Analysis", "icon": "\U0001fa7a", "color": "#10B981"},
    {"id": 6, "label": "Disease Prediction", "icon": "\U0001f3af", "color": "#EF4444"},
    {"id": 7, "label": "Differential Dx", "icon": "\U0001f4ca", "color": "#6366F1"},
    {"id": 8, "label": "Context Memory", "icon": "\U0001f9e0", "color": "#14B8A6"},
    {"id": 9, "label": "Decision Engine", "icon": "\u26a1", "color": "#F97316"},
    {"id": 10, "label": "Treatment Plan", "icon": "\U0001f48a", "color": "#06B6D4"},
    {"id": 11, "label": "LLM Reasoning", "icon": "\U0001f916", "color": "#A855F7"},
    {"id": 12, "label": "Feedback Loop", "icon": "\U0001f501", "color": "#84CC16"},
    {"id": 13, "label": "Output", "icon": "\U0001f4cb", "color": "#0EA5E9"},
]

SECTION_COLORS = ["#0F766E", "#0EA5E9", "#8B5CF6", "#EC4899", "#F59E0B",
                  "#EF4444", "#6366F1", "#14B8A6", "#F97316"]
SECTION_ICONS = ["\U0001f4cb", "\U0001f50d", "\U0001f3af", "\u26a0\ufe0f", "\U0001f4a1", "\U0001f52c", "\U0001f4ca", "\U0001fa7a", "\U0001f48a"]


# ══════════════════════════════════════════════════════════════
# SYSTEM PROMPTS
# ══════════════════════════════════════════════════════════════

SYSTEM_PROMPTS = {
    "image_diagnosis": """You are MediAI, an advanced healthcare diagnosis intelligence agent built by Aigenthix. You assist healthcare professionals with medical image interpretation.

IMPORTANT DISCLAIMER: You are an AI assistant and NOT a licensed medical professional. All outputs are for informational and educational support only. Always recommend consulting a qualified healthcare provider.

When analyzing medical images, provide:
## Image Assessment
Describe image type and quality observations
## Findings
Possible abnormalities or notable features (regions, density, symmetry)
## Differential Diagnosis
List 3-5 possible conditions ranked by likelihood with confidence %
## Risk Score
Overall risk (Low/Moderate/High/Critical) with numeric score 0-100
## Recommended Next Steps
Additional tests, imaging, specialist consultations
## Clinical Correlation
How findings correlate with patient data
## Urgency Level
Whether immediate attention is required

Use ## headers for each section. Use medical terminology with plain-language explanations.""",

    "symptom_diagnosis": """You are MediAI, an advanced clinical diagnosis intelligence agent by Aigenthix. You help healthcare professionals with symptom-based differential diagnosis.

DISCLAIMER: AI assistant only. Not a substitute for professional medical judgment.

Given patient symptoms, history, and vitals, provide using ## headers:
## Clinical Assessment Summary
## Symptom Analysis
## Differential Diagnosis
Top 5 possible conditions with probability %
## Risk Stratification
Low/Moderate/High/Critical with numeric score 0-100
## Recommended Diagnostic Workup
## Red Flags
## Initial Management Suggestions
## Specialist Referral Recommendations

Be thorough, evidence-based, and clinically structured.""",

    "lab_report": """You are MediAI Lab Report Analyst by Aigenthix.

DISCLAIMER: AI-assisted interpretation only.

Given lab values, provide using ## headers:
## Abnormal Values Summary
## Pattern Recognition
## Clinical Significance
## Differential Diagnosis
## Risk Markers
Include risk score 0-100
## Recommended Follow-up Tests
## Clinical Correlation Notes""",

    "chronic_disease": """You are MediAI Chronic Disease Monitor by Aigenthix.

DISCLAIMER: AI monitoring support only.

Given chronic disease data, provide using ## headers:
## Disease Status Assessment
## Trend Analysis
## Risk Alerts
Include risk score 0-100
## Complication Risk
## Management Optimization
## Lifestyle Recommendations
## Medication Considerations
## Follow-up Timeline""",

    "treatment_plan": """You are MediAI Treatment Planning Support by Aigenthix.

DISCLAIMER: AI-generated treatment support only.

Given diagnosis and patient profile, provide using ## headers:
## Treatment Pathway Overview
## First-Line Interventions
## Alternative Options
## Specialist Referral Pathway
## Monitoring Plan
## Expected Timeline
## Risk-Benefit Analysis
## Patient Education Points
## Case Summary""",

    "patient_explain": """You are MediAI Patient Communication Assistant by Aigenthix.

Translate clinical findings into patient-friendly language using ## headers:
## Simple Summary
8th-grade reading level
## What This Means
## What Happens Next
## Questions to Ask Your Doctor
## Lifestyle Tips
## When to Seek Help
## Support Resources

Warm, reassuring tone. No jargon. Use analogies.
If a language other than English is requested, write the ENTIRE response in that language."""
}


# ══════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS
# ══════════════════════════════════════════════════════════════

def extract_risk_score(text):
    if not text: return 35
    m = re.search(r'(\d{1,3})(?:\s*/\s*100|\s*%|\s*out of 100)', text)
    if m: return min(int(m.group(1)), 100)
    if re.search(r'critical', text, re.I): return 90
    if re.search(r'high', text, re.I): return 72
    if re.search(r'moderate', text, re.I): return 48
    if re.search(r'low', text, re.I): return 22
    return 35

def get_risk_color(s):
    if s >= 75: return "#DC2626"
    if s >= 50: return "#F59E0B"
    if s >= 25: return "#3B82F6"
    return "#10B981"

def get_risk_label(s):
    if s >= 75: return "Critical"
    if s >= 50: return "Moderate"
    if s >= 25: return "Low"
    return "Minimal"

def get_risk_emoji(s):
    if s >= 75: return "\U0001f534"
    if s >= 50: return "\U0001f7e1"
    if s >= 25: return "\U0001f535"
    return "\U0001f7e2"

def parse_sections(text):
    if not text: return []
    lines = text.split("\n")
    sections, cur = [], None
    for line in lines:
        hm = re.match(r'^#{1,3}\s+\**(.+?)\**\s*$', line) or re.match(r'^#{1,3}\s+(.+)$', line)
        bm = re.match(r'^\**\d+\.\s*(.+?)\**\s*[\u2014\u2013-]', line) or re.match(r'^\**(.+?)\**\s*[\u2014\u2013-]', line)
        if hm:
            if cur: sections.append(cur)
            cur = {"title": hm.group(1).replace("*",""), "content": []}
        elif bm and not cur:
            cur = {"title": bm.group(1).replace("*",""), "content": []}
        elif cur:
            cur["content"].append(line)
        elif not sections and line.strip():
            cur = {"title": "Overview", "content": [line]}
    if cur: sections.append(cur)
    return sections


def build_risk_html(score):
    c = get_risk_color(score)
    lb = get_risk_label(score)
    dg = score * 3.6
    st = "Urgent" if score >= 75 else ("Monitor" if score >= 50 else "Stable")
    cf = "High" if score >= 60 else ("Moderate" if score >= 30 else "Preliminary")
    return f'''<div style="display:flex;align-items:center;gap:20px;flex-wrap:wrap;">
      <div style="width:120px;height:120px;border-radius:50%;background:conic-gradient({c} {dg}deg,#E2E8F0 0deg);display:flex;align-items:center;justify-content:center;flex-shrink:0;">
        <div style="width:90px;height:90px;border-radius:50%;background:#fff;display:flex;flex-direction:column;align-items:center;justify-content:center;">
          <span style="font-size:28px;font-weight:800;color:{c};font-family:'DM Sans',sans-serif;">{score}</span>
          <span style="font-size:10px;color:#64748B;font-weight:600;">/ 100</span>
        </div>
      </div>
      <div style="flex-shrink:0;">
        <div style="font-size:18px;font-weight:700;color:{c};font-family:'DM Sans',sans-serif;">{lb} Risk</div>
        <div style="font-size:12px;color:#64748B;margin-top:4px;">AI-computed risk assessment</div>
      </div>
      <div style="flex:1;display:grid;grid-template-columns:1fr 1fr;gap:10px;min-width:220px;">
        <div style="padding:14px;background:linear-gradient(135deg,#0F766E08,#0F766E15);border-radius:14px;border:1px solid #0F766E30;text-align:center;">
          <div style="font-size:11px;color:#64748B;font-weight:600;">Status</div>
          <div style="font-size:16px;font-weight:800;color:#0F766E;margin-top:4px;">{st}</div>
        </div>
        <div style="padding:14px;background:linear-gradient(135deg,#8B5CF608,#8B5CF615);border-radius:14px;border:1px solid #8B5CF630;text-align:center;">
          <div style="font-size:11px;color:#64748B;font-weight:600;">Confidence</div>
          <div style="font-size:16px;font-weight:800;color:#8B5CF6;margin-top:4px;">{cf}</div>
        </div>
      </div>
    </div>'''


def build_sections_html(text):
    secs = parse_sections(text)
    if not secs:
        cl = re.sub(r'\*\*','',text or '')
        cl = re.sub(r'#{1,3}\s','',cl)
        cl = cl.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        return f'<div style="padding:16px;background:#F8FAFC;border-radius:12px;border-left:4px solid #0F766E;font-size:13px;line-height:1.7;color:#334155;white-space:pre-wrap;">{cl}</div>'
    html = ""
    for i, s in enumerate(secs):
        c = SECTION_COLORS[i % len(SECTION_COLORS)]
        ic = SECTION_ICONS[i % len(SECTION_ICONS)]
        ct = "\n".join(s["content"]).replace("**","").strip()
        ct = ct.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        html += f'''<div style="padding:16px;background:#F8FAFC;border-radius:12px;margin-bottom:12px;border-left:4px solid {c};">
          <div style="font-size:14px;font-weight:700;margin-bottom:8px;color:{c};">{ic} {s["title"]}</div>
          <div style="font-size:13px;color:#334155;line-height:1.7;white-space:pre-wrap;">{ct}</div>
        </div>'''
    return html


def build_output_html(text, score, atype="Diagnosis", wlog=None):
    secs = parse_sections(text)
    ns = len(secs) if secs else 1
    log_html = ""
    if wlog:
        ll = "\n".join(wlog)
        log_html = f'''<div style="background:#fff;border-radius:16px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06);border:1px solid #E2E8F0;margin-top:16px;">
          <div style="font-size:14px;font-weight:700;margin-bottom:12px;color:#0F172A;font-family:'DM Sans',sans-serif;">\U0001f501 Workflow Log</div>
          <pre style="font-size:11px;color:#64748B;background:#F8FAFC;padding:12px;border-radius:10px;overflow-x:auto;white-space:pre-wrap;margin:0;">{ll}</pre>
        </div>'''
    return f'''<div>
      <div style="background:#fff;border-radius:16px;padding:24px;box-shadow:0 1px 3px rgba(0,0,0,0.06);border:1px solid #E2E8F0;margin-bottom:16px;">
        <div style="font-size:16px;font-weight:700;margin-bottom:16px;color:#0F172A;font-family:'DM Sans',sans-serif;">\U0001f4ca Assessment Overview</div>
        {build_risk_html(score)}
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-top:16px;">
          <div style="padding:14px;background:linear-gradient(135deg,#EC489908,#EC489915);border-radius:14px;border:1px solid #EC489930;text-align:center;">
            <div style="font-size:11px;color:#64748B;font-weight:600;">Analysis</div>
            <div style="font-size:16px;font-weight:800;color:#EC4899;margin-top:4px;">{atype}</div>
          </div>
          <div style="padding:14px;background:linear-gradient(135deg,#F59E0B08,#F59E0B15);border-radius:14px;border:1px solid #F59E0B30;text-align:center;">
            <div style="font-size:11px;color:#64748B;font-weight:600;">Sections</div>
            <div style="font-size:16px;font-weight:800;color:#F59E0B;margin-top:4px;">{ns}</div>
          </div>
        </div>
      </div>
      <div style="background:#fff;border-radius:16px;padding:24px;box-shadow:0 1px 3px rgba(0,0,0,0.06);border:1px solid #E2E8F0;">
        <div style="font-size:16px;font-weight:700;margin-bottom:16px;color:#0F172A;font-family:'DM Sans',sans-serif;">\U0001f3e5 Diagnostic Report</div>
        <div style="max-height:600px;overflow-y:auto;padding-right:8px;">{build_sections_html(text)}</div>
      </div>
      {log_html}
    </div>'''

print("\u2705 Constants, prompts & utilities loaded.")

✅ Constants, prompts & utilities loaded.


## 🖼️ Step 4: Medical Image Processor + Groq LLM Engine

In [4]:
class MedicalImageProcessor:
    @staticmethod
    def preprocess(image) -> dict:
        if image is None: return {"error": "No image"}
        img = np.array(image) if not isinstance(image, np.ndarray) else image
        if len(img.shape) == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4: img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mi = float(np.mean(gray)); si = float(np.std(gray))
        edges = cv2.Canny(gray, 50, 150)
        ed = float(np.sum(edges > 0) / (h * w))
        cs = float(np.std(img.reshape(-1, 3).std(axis=1)))
        h2, w2 = h//2, w//2
        left = gray[:, :w2]; right = cv2.flip(gray[:, w2:], 1)
        mw = min(left.shape[1], right.shape[1])
        sym = float(1.0 - np.mean(np.abs(left[:,:mw].astype(float) - right[:,:mw].astype(float))) / 255.0)
        lv = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        qi = []
        if mi < 30: qi.append("Underexposed")
        if mi > 225: qi.append("Overexposed")
        if si < 20: qi.append("Low contrast")
        if lv < 50: qi.append("Possible blur")
        if h < 200 or w < 200: qi.append("Low resolution")
        return {"dims": f"{w}x{h}", "mean": round(mi,2), "std": round(si,2),
            "contrast": round(si/(mi+1e-6),4), "edge_density": round(ed,4),
            "symmetry": round(sym,4), "grayscale": cs < 15,
            "texture": round(lv,2), "quality": "Good" if not qi else "; ".join(qi),
            "quads": {k: round(float(np.mean(gray[s])),2) for k,s in
                [("TL",np.s_[:h2,:w2]),("TR",np.s_[:h2,w2:]),("BL",np.s_[h2:,:w2]),("BR",np.s_[h2:,w2:])]}}

    @staticmethod
    def to_text(f):
        if "error" in f: return f["error"]
        return f"""IMAGE FEATURES: {f['dims']}, {'Grayscale' if f['grayscale'] else 'Color'}, Mean:{f['mean']}, Contrast:{f['contrast']}, Edges:{f['edge_density']}, Symmetry:{f['symmetry']}, Texture:{f['texture']}, Quality:{f['quality']}, Quads:{json.dumps(f['quads'])}"""


class GroqLLMEngine:
    def __init__(self):
        self.client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    def generate(self, sys, msg):
        try:
            r = self.client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role":"system","content":sys},{"role":"user","content":msg}],
                max_tokens=4096, temperature=0.3, top_p=0.9)
            return r.choices[0].message.content
        except Exception as e: return f"\u274c LLM Error: {e}"

imgp = MedicalImageProcessor()
llm = GroqLLMEngine()
print("\u2705 Image Processor + Groq LLM ready.")

✅ Image Processor + Groq LLM ready.


## 🔁 Step 5: LangGraph 13-Node Agent Workflow

In [5]:
class AgentState(TypedDict):
    image: Optional[Any]; image_type: str; patient_name: str; age: str; gender: str
    blood_pressure: str; heart_rate: str; temperature: str; spo2: str
    medical_history: str; medications: str; symptoms: str; lab_values: str
    chronic_conditions: str; chronic_data: str; specialty: str; diagnosis_type: str
    analysis_mode: str; treatment_notes: str; clinical_findings: str; language: str
    feedback: str; image_features: str; patient_context: str; extracted_features: str
    memory_context: str; llm_response: str; final_output: str; risk_score: int; workflow_log: list

def _ts(): return datetime.now().strftime('%H:%M:%S')

def n1(s):
    l = s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 1: Patient input received"); return {"workflow_log":l}

def n2(s):
    l = s.get("workflow_log",[])
    ctx = [f"{lb}: {s[k]}" for k,lb in [("patient_name","Patient"),("age","Age"),("gender","Gender"),("blood_pressure","BP"),("heart_rate","HR"),("temperature","Temp"),("spo2","SpO2"),("medical_history","History"),("medications","Meds"),("specialty","Specialty"),("diagnosis_type","DxType")] if s.get(k)]
    l.append(f"[{_ts()}] \u2705 Node 2: Preprocessed"); return {"patient_context":"\n".join(ctx) or "No data.","workflow_log":l}

def n3(s):
    l = s.get("workflow_log",[])
    if s.get("image") is not None:
        try:
            f = imgp.preprocess(s["image"]); ft = imgp.to_text(f)
            l.append(f"[{_ts()}] \u2705 Node 3: Image analyzed"); return {"image_features":ft,"workflow_log":l}
        except: pass
    l.append(f"[{_ts()}] \u2139\ufe0f Node 3: No image"); return {"image_features":"No image.","workflow_log":l}

def n4(s):
    l = s.get("workflow_log",[])
    p = []
    if s.get("image_features") and "No image" not in s["image_features"]: p.append(f"IMAGE:\n{s['image_features']}")
    if s.get("symptoms"): p.append(f"SYMPTOMS:\n{s['symptoms']}")
    if s.get("lab_values"): p.append(f"LAB:\n{s['lab_values']}")
    if s.get("chronic_conditions"): p.append(f"CHRONIC:\n{s['chronic_conditions']}")
    if s.get("chronic_data"): p.append(f"DATA:\n{s['chronic_data']}")
    l.append(f"[{_ts()}] \u2705 Node 4: Features ({len(p)} sources)"); return {"extracted_features":"\n\n".join(p) or "None.","workflow_log":l}

def n5(s): l=s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 5: Symptom analysis"); return {"workflow_log":l}
def n6(s): l=s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 6: Disease prediction"); return {"workflow_log":l}
def n7(s): l=s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 7: Differential Dx"); return {"workflow_log":l}

def n8(s):
    l = s.get("workflow_log",[])
    p = [f"PATIENT:\n{s.get('patient_context','N/A')}",f"FEATURES:\n{s.get('extracted_features','N/A')}"]
    if s.get("feedback"): p.append(f"FEEDBACK:\n{s['feedback']}")
    l.append(f"[{_ts()}] \u2705 Node 8: Context assembled"); return {"memory_context":"\n\n".join(p),"workflow_log":l}

def n9(s): l=s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 9: Decision engine"); return {"workflow_log":l}
def n10(s): l=s.get("workflow_log",[]); l.append(f"[{_ts()}] \u2705 Node 10: Treatment ready"); return {"workflow_log":l}

def n11(s):
    l = s.get("workflow_log",[]); l.append(f"[{_ts()}] \U0001f916 Node 11: LLM reasoning...")
    mode = s.get("analysis_mode","symptom")
    pm = {"image":"image_diagnosis","symptom":"symptom_diagnosis","lab":"lab_report","chronic":"chronic_disease","treatment":"treatment_plan","explain":"patient_explain"}
    sp = SYSTEM_PROMPTS.get(pm.get(mode,"symptom_diagnosis"))
    up = [s.get("memory_context","")]
    if mode=="image": up.append(f"\nImage Type: {s.get('image_type','N/A')}\nProvide analysis with risk score/100.")
    elif mode=="symptom": up.append("\nDifferential diagnosis with risk score/100.")
    elif mode=="lab": up.append("\nInterpret labs with risk score/100.")
    elif mode=="chronic": up.append("\nChronic assessment with risk score/100.")
    elif mode=="treatment":
        if s.get("treatment_notes"): up.append(f"\nNotes:\n{s['treatment_notes']}")
        up.append("\nGenerate treatment plan.")
    elif mode=="explain":
        f = s.get("clinical_findings","") or s.get("llm_response","")
        up = [f"Translate to patient-friendly in {s.get('language','English')}:\n\n{f}"]
    r = llm.generate(sp, "\n".join(up)); sc = extract_risk_score(r)
    l.append(f"[{_ts()}] \u2705 Node 11: LLM done"); return {"llm_response":r,"risk_score":sc,"workflow_log":l}

def n12(s):
    l=s.get("workflow_log",[])
    l.append(f"[{_ts()}] {'\U0001f501 Node 12: Feedback used' if s.get('feedback') else '\u2705 Node 12: No feedback'}")
    return {"workflow_log":l}

def n13(s):
    l=s.get("workflow_log",[])
    l.append(f"[{_ts()}] \u2705 Node 13: Output delivered")
    l.append(f"[{_ts()}] \U0001f3c1 Workflow complete.")
    return {"final_output":s.get("llm_response","No output."),"workflow_log":l}

def build_graph():
    g = StateGraph(AgentState)
    for i,fn in enumerate([n1,n2,n3,n4,n5,n6,n7,n8,n9,n10,n11,n12,n13],1):
        g.add_node(f"n{i}", fn)
    g.set_entry_point("n1")
    for i in range(1,13): g.add_edge(f"n{i}",f"n{i+1}")
    g.add_edge("n13",END)
    return g.compile()

graph = build_graph()
print("\u2705 LangGraph workflow built.")

✅ LangGraph workflow built.


In [6]:
class MediAIAgent:
    def __init__(self):
        self.graph = graph; self.last_result = None
    def run(self, **kw):
        d = {k:"" for k in ["image_type","patient_name","age","gender","blood_pressure","heart_rate","temperature","spo2","medical_history","medications","symptoms","lab_values","chronic_conditions","chronic_data","specialty","diagnosis_type","analysis_mode","treatment_notes","clinical_findings","language","feedback","image_features","patient_context","extracted_features","memory_context","llm_response","final_output"]}
        d.update({"image":None,"risk_score":0,"workflow_log":[],"analysis_mode":"symptom","language":"English"})
        d.update(kw)
        r = self.graph.invoke(d); self.last_result = r; return r

agent = MediAIAgent()
print("\u2705 Agent ready.")

✅ Agent ready.


## 🎨 Step 7: Gradio UI — Exact Match to React Design

In [7]:
import gradio as gr

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');

.gradio-container{max-width:1400px!important;font-family:'DM Sans','Segoe UI',sans-serif!important;background:#F0F4F8!important;color:#1E293B!important}
.dark .gradio-container,.dark{background:#F0F4F8!important}

/* Cards */
.gr-group,.gr-box,.gr-panel{background:#fff!important;border-radius:16px!important;border:1px solid #E2E8F0!important;box-shadow:0 1px 3px rgba(0,0,0,.06),0 1px 2px rgba(0,0,0,.04)!important}

/* Inputs */
input[type=text],input[type=number],textarea,.gr-input,.gr-text-input{background:#F8FAFC!important;border:1.5px solid #CBD5E1!important;border-radius:10px!important;font-size:14px!important;padding:10px 14px!important;font-family:'DM Sans',sans-serif!important;color:#1E293B!important}
input:focus,textarea:focus{border-color:#0F766E!important;box-shadow:0 0 0 3px rgba(15,118,110,.1)!important}

/* Selects */
select,.gr-dropdown{background:#F8FAFC!important;border:1.5px solid #CBD5E1!important;border-radius:10px!important;font-size:14px!important}

/* Labels */
label,.gr-label{font-size:13px!important;font-weight:600!important;color:#475569!important;font-family:'DM Sans',sans-serif!important}

/* Primary btn: gradient */
.gr-button-primary,button.primary{background:linear-gradient(135deg,#0F766E,#0EA5E9)!important;color:#fff!important;border:none!important;border-radius:12px!important;font-size:14px!important;font-weight:700!important;padding:12px 28px!important;box-shadow:0 2px 8px rgba(15,118,110,.3)!important;font-family:'DM Sans',sans-serif!important;transition:all .2s!important}
.gr-button-primary:hover,button.primary:hover{opacity:.92!important;transform:translateY(-1px)!important}

/* Secondary btn */
.gr-button-secondary,button.secondary{background:#F1F5F9!important;color:#475569!important;border:1.5px solid #CBD5E1!important;border-radius:10px!important;font-size:13px!important;font-weight:600!important}

/* Stop btn */
.gr-button-stop{background:#FEF2F2!important;color:#DC2626!important;border:1.5px solid #FECACA!important;border-radius:10px!important}

/* Tabs: active #0F766E bottom border */
.tabs .tab-nav button{font-family:'DM Sans',sans-serif!important;font-size:13px!important;font-weight:500!important;color:#64748B!important;padding:14px 18px!important;border:none!important;border-bottom:3px solid transparent!important;background:transparent!important;transition:all .2s!important;white-space:nowrap!important}
.tabs .tab-nav button.selected{font-weight:700!important;color:#0F766E!important;background:rgba(15,118,110,.06)!important;border-bottom:3px solid #0F766E!important}
.tabs .tab-nav{background:#fff!important;border-bottom:1px solid #E2E8F0!important;gap:2px!important;padding:0 24px!important}

/* Checkbox */
.gr-check-radio label{padding:10px 14px!important;border-radius:10px!important;border:1.5px solid #E2E8F0!important;background:#F8FAFC!important;font-size:13px!important;font-weight:500!important}
.gr-check-radio label:has(input:checked){background:#F0FDF4!important;border-color:#86EFAC!important}

/* Image upload */
.gr-image-upload,.upload-zone{border:2px dashed #CBD5E1!important;border-radius:14px!important;background:#F8FAFC!important}

/* Disclaimer */
#disclaimer-box{background:#FFFBEB;border:1px solid #FDE68A;border-radius:10px;padding:14px 18px;font-size:12px;color:#92400E;line-height:1.5;margin-bottom:16px}

/* Scrollbar */
::-webkit-scrollbar{width:6px;height:6px}::-webkit-scrollbar-thumb{background:#CBD5E1;border-radius:10px}

/* Animations */
@keyframes spin{to{transform:rotate(360deg)}}
@keyframes fadeIn{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
@keyframes marquee{0%{transform:translateX(0%)}100%{transform:translateX(-50%)}}
"""

HEADER = """
<div style="background:linear-gradient(135deg,#0F172A 0%,#1E3A5F 50%,#0F766E 100%);padding:0;position:relative;overflow:hidden;border-radius:16px 16px 0 0;margin:-16px -16px 16px -16px;">
  <div style="position:absolute;inset:0;opacity:.05;background-image:radial-gradient(circle at 20% 80%,#0EA5E9 1px,transparent 1px),radial-gradient(circle at 80% 20%,#10B981 1px,transparent 1px);background-size:30px 30px;"></div>
  <div style="background:rgba(255,255,255,.08);backdrop-filter:blur(10px);padding:8px 0;overflow:hidden;white-space:nowrap;font-size:12px;color:rgba(255,255,255,.7);letter-spacing:.5px;">
    <span style="display:inline-block;animation:marquee 20s linear infinite;">\u25c6&nbsp;MediAI: Healthcare Diagnosis Intelligence Agent&nbsp;\u25c6&nbsp;Powered by Aigenthix&nbsp;\u25c6&nbsp;AI-Assisted Diagnostic Support&nbsp;\u25c6&nbsp;Medical Image Analysis&nbsp;\u25c6&nbsp;Clinical Decision Intelligence&nbsp;\u25c6&nbsp;Next-Generation Healthcare AI&nbsp;\u25c6&nbsp;Real-Time Diagnosis Support&nbsp;\u25c6&nbsp;Treatment Planning Assistant&nbsp;\u25c6&nbsp;MediAI: Healthcare Diagnosis Intelligence Agent&nbsp;\u25c6&nbsp;Powered by Aigenthix&nbsp;\u25c6&nbsp;AI-Assisted Diagnostic Support&nbsp;\u25c6&nbsp;Medical Image Analysis&nbsp;\u25c6&nbsp;Clinical Decision Intelligence&nbsp;\u25c6&nbsp;</span>
  </div>
  <div style="padding:20px 32px;position:relative;z-index:2;">
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <div>
        <h1 style="font-size:28px;font-weight:800;color:#fff;margin:0;letter-spacing:-.5px;font-family:'DM Sans',sans-serif;">\U0001f3e5 MediAI<span style="display:inline-block;background:rgba(16,185,129,.2);border:1px solid rgba(16,185,129,.3);color:#34D399;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:600;margin-left:12px;">v2.0</span></h1>
        <p style="font-size:13px;color:rgba(255,255,255,.6);margin:4px 0 0 0;font-weight:400;font-family:'DM Sans',sans-serif;">Healthcare Diagnosis Intelligence Agent by Aigenthix</p>
      </div>
      <div style="display:flex;gap:8px;align-items:center;"><div style="width:8px;height:8px;border-radius:50%;background:#34D399;box-shadow:0 0 8px #34D399;"></div><span style="font-size:12px;color:rgba(255,255,255,.6);font-family:'DM Sans',sans-serif;">AI Engine Active</span></div>
    </div>
  </div>
</div>
"""

DISCL = '<div id="disclaimer-box">\u26a0\ufe0f <strong>Medical Disclaimer:</strong> MediAI is an AI-assisted tool for informational and educational support only. It is NOT a substitute for professional medical diagnosis, advice, or treatment. Always consult a qualified healthcare provider.</div>'

FOOTER = '<div style="text-align:center;padding:32px 0 16px;color:#94A3B8;font-size:12px;font-family:\'DM Sans\',sans-serif;"><div style="font-weight:700;color:#64748B;margin-bottom:4px;">MediAI Healthcare Diagnosis Intelligence Agent</div><div>Built by Aigenthix \u2022 LangGraph + Groq (llama-3.1-8b-instant) + Gradio \u2022 AI-Assisted Medical Decision Support</div></div>'

print("\u2705 UI templates ready.")

✅ UI templates ready.


In [8]:
# ══════════════════════════════════════════════════════════════
# HANDLERS — one per tab, matching React exactly
# ══════════════════════════════════════════════════════════════

def _err(msg): return f'<div style="padding:24px;text-align:center;color:#94A3B8;font-family:\'DM Sans\',sans-serif;">\u26a0\ufe0f {msg}</div>'

def h_image(img,itype,spec,dx,nm,ag,gn,bp,hr,tp,sp,hi,md,sym):
    if img is None and not sym: return _err("Please upload an image or provide symptoms.")
    r=agent.run(image=img,image_type=itype or"",specialty=spec or"",diagnosis_type=dx or"",patient_name=nm or"",age=ag or"",gender=gn or"",blood_pressure=bp or"",heart_rate=hr or"",temperature=tp or"",spo2=sp or"",medical_history=hi or"",medications=md or"",symptoms=sym or"",analysis_mode="image")
    return build_output_html(r["final_output"],r["risk_score"],"Image",r.get("workflow_log"))

def h_symptom(spec,dx,nm,ag,gn,bp,hr,tp,sp,hi,md,sym):
    if not sym: return _err("Please enter symptoms.")
    r=agent.run(specialty=spec or"",diagnosis_type=dx or"",patient_name=nm or"",age=ag or"",gender=gn or"",blood_pressure=bp or"",heart_rate=hr or"",temperature=tp or"",spo2=sp or"",medical_history=hi or"",medications=md or"",symptoms=sym,analysis_mode="symptom")
    return build_output_html(r["final_output"],r["risk_score"],"Diagnosis",r.get("workflow_log"))

def h_lab(nm,ag,gn,bp,hr,tp,sp,hi,md,lv):
    if not lv: return _err("Please enter lab values.")
    r=agent.run(patient_name=nm or"",age=ag or"",gender=gn or"",blood_pressure=bp or"",heart_rate=hr or"",temperature=tp or"",spo2=sp or"",medical_history=hi or"",medications=md or"",lab_values=lv,analysis_mode="lab")
    return build_output_html(r["final_output"],r["risk_score"],"Lab Report",r.get("workflow_log"))

def h_chronic(nm,ag,gn,bp,hr,tp,sp,hi,md,cond,cd):
    if not cond and not cd: return _err("Please select conditions or enter data.")
    cs=", ".join(cond) if isinstance(cond,list) else str(cond or "")
    r=agent.run(patient_name=nm or"",age=ag or"",gender=gn or"",blood_pressure=bp or"",heart_rate=hr or"",temperature=tp or"",spo2=sp or"",medical_history=hi or"",medications=md or"",chronic_conditions=cs,chronic_data=cd or"",analysis_mode="chronic")
    return build_output_html(r["final_output"],r["risk_score"],"Chronic",r.get("workflow_log"))

def h_treatment(nm,ag,gn,bp,hr,tp,sp,hi,md,nt):
    src=nt
    if not src and agent.last_result: src=agent.last_result.get("final_output","")
    if not src: return _err("Please provide notes or run an analysis first.")
    r=agent.run(patient_name=nm or"",age=ag or"",gender=gn or"",blood_pressure=bp or"",heart_rate=hr or"",temperature=tp or"",spo2=sp or"",medical_history=hi or"",medications=md or"",treatment_notes=src,analysis_mode="treatment")
    return build_output_html(r["final_output"],r.get("risk_score",0),"Treatment",r.get("workflow_log"))

def h_summary():
    if not agent.last_result: return _err("Run an analysis first.")
    p=agent.last_result.get("final_output","")
    resp=llm.generate(SYSTEM_PROMPTS["treatment_plan"],f"Generate clinical summary:\n\n{p}")
    return build_output_html(resp,extract_risk_score(resp),"Summary",None)

def h_explain(fi,la):
    src=fi
    if not src and agent.last_result: src=agent.last_result.get("final_output","")
    if not src: return _err("Please provide findings or run analysis first.")
    r=agent.run(clinical_findings=src,language=la or "English",analysis_mode="explain")
    return build_output_html(r["final_output"],0,"Explanation",r.get("workflow_log"))

def h_refine(fb):
    if not fb: return _err("Please enter feedback.")
    if not agent.last_result: return _err("Run an analysis first.")
    p=agent.last_result.get("final_output","")
    resp=llm.generate(SYSTEM_PROMPTS["symptom_diagnosis"],f"Previous:\n{p}\n\nFeedback:\n{fb}\n\nRefine.")
    sc=extract_risk_score(resp)
    if agent.last_result: agent.last_result["final_output"]=resp
    return build_output_html(resp,sc,"Refined",None)

print("\u2705 Handlers ready.")

✅ Handlers ready.


In [9]:
# ══════════════════════════════════════════════════════════════
# BUILD GRADIO APP
# ══════════════════════════════════════════════════════════════

def pf():
    """Patient form — matches React PatientDataForm."""
    gr.Markdown("#### \U0001f464 Patient Information")
    with gr.Row():
        nm=gr.Textbox(label="Patient Name",placeholder="Full name",scale=2)
        ag=gr.Textbox(label="Age",placeholder="Age",scale=1)
        gn=gr.Dropdown(["Male","Female","Other"],label="Gender",scale=1)
    gr.Markdown("#### \U0001f4ca Vitals")
    with gr.Row():
        bp=gr.Textbox(label="Blood Pressure",placeholder="120/80 mmHg")
        hr=gr.Textbox(label="Heart Rate",placeholder="72 bpm")
        tp=gr.Textbox(label="Temperature",placeholder="98.6\u00b0F")
        sp=gr.Textbox(label="SpO2",placeholder="98%")
    with gr.Row():
        hi=gr.Textbox(label="Medical History",placeholder="Past conditions, surgeries, allergies...",lines=2)
        md=gr.Textbox(label="Current Medications",placeholder="List medications",lines=2)
    return nm,ag,gn,bp,hr,tp,sp,hi,md


with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="MediAI by Aigenthix") as app:
    gr.HTML(HEADER)
    gr.HTML(DISCL)

    with gr.Tabs():
        # Tab 1: Image
        with gr.TabItem("\U0001f52c Image Diagnosis"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f52c Medical Image Analysis")
                    iu=gr.Image(label="Upload Medical Image",type="numpy")
                    with gr.Row():
                        it=gr.Dropdown(IMAGE_TYPES,label="Image Type")
                        isp=gr.Dropdown(SPECIALTIES,label="Specialty")
                    idx=gr.Dropdown(DIAGNOSIS_TYPES,label="Diagnosis Type")
                    gr.Markdown("### \U0001f4dd Additional Context")
                    in_,ia,ig,ibp,ihr,itp,isp2,ih,im=pf()
                    isym=gr.Textbox(label="Clinical Notes / Symptoms",placeholder="Additional observations...",lines=3)
                    with gr.Row():
                        ib=gr.Button("\U0001f52c Analyze Image",variant="primary",size="lg")
                        ic=gr.Button("\U0001f5d1\ufe0f Clear",variant="secondary")
                with gr.Column(scale=1):
                    io=gr.HTML()
            ib.click(h_image,[iu,it,isp,idx,in_,ia,ig,ibp,ihr,itp,isp2,ih,im,isym],[io])
            ic.click(lambda:"",outputs=[io])

        # Tab 2: Symptom
        with gr.TabItem("\U0001fa7a Symptom Diagnosis"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001fa7a Symptom-Based Diagnosis")
                    with gr.Row():
                        ss=gr.Dropdown(SPECIALTIES,label="Specialty Focus")
                        sd=gr.Dropdown(DIAGNOSIS_TYPES,label="Diagnosis Type")
                    sn,sa,sg,sbp,shr,stp,ssp,sh,sm=pf()
                    ssym=gr.Textbox(label="Presenting Symptoms *",placeholder="Onset, duration, severity, location...",lines=5)
                    with gr.Row():
                        sb=gr.Button("\U0001fa7a Analyze Symptoms",variant="primary",size="lg")
                        sc2=gr.Button("\U0001f5d1\ufe0f Clear",variant="secondary")
                with gr.Column(scale=1):
                    so=gr.HTML()
            sb.click(h_symptom,[ss,sd,sn,sa,sg,sbp,shr,stp,ssp,sh,sm,ssym],[so])
            sc2.click(lambda:"",outputs=[so])

        # Tab 3: Lab
        with gr.TabItem("\U0001f9ea Lab Reports"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f9ea Lab Report Interpretation")
                    ln,la2,lg2,lbp,lhr,ltp,lsp,lh,lm=pf()
                    lv=gr.Textbox(label="Lab Values *",placeholder="Hemoglobin: 10.2 g/dL\nWBC: 12,500 /\u00b5L\nFasting Glucose: 142 mg/dL\nHbA1c: 7.8%\nTSH: 6.2 mIU/L",lines=8)
                    with gr.Row():
                        lb2=gr.Button("\U0001f9ea Interpret Report",variant="primary",size="lg")
                        lc=gr.Button("\U0001f5d1\ufe0f Clear",variant="secondary")
                with gr.Column(scale=1):
                    lo=gr.HTML()
            lb2.click(h_lab,[ln,la2,lg2,lbp,lhr,ltp,lsp,lh,lm,lv],[lo])
            lc.click(lambda:"",outputs=[lo])

        # Tab 4: Chronic
        with gr.TabItem("\u2764\ufe0f Chronic Disease"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \u2764\ufe0f Chronic Disease Monitoring")
                    cn,ca,cg,cbp,chr2,ctp,csp,ch2,cm=pf()
                    cc=gr.CheckboxGroup(CHRONIC_CONDITIONS,label="Select Chronic Conditions")
                    cd=gr.Textbox(label="Recent Values / Observations",placeholder="Blood sugar trends, BP logs, weight changes...",lines=4)
                    with gr.Row():
                        cb=gr.Button("\u2764\ufe0f Assess Risk",variant="primary",size="lg")
                        cc2=gr.Button("\U0001f5d1\ufe0f Clear",variant="secondary")
                with gr.Column(scale=1):
                    co=gr.HTML()
            cb.click(h_chronic,[cn,ca,cg,cbp,chr2,ctp,csp,ch2,cm,cc,cd],[co])
            cc2.click(lambda:"",outputs=[co])

        # Tab 5: Treatment
        with gr.TabItem("\U0001f48a Treatment Plan"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f48a Treatment Planning Support")
                    tn,ta,tg,tbp,thr,ttp,tsp,th,tm=pf()
                    tnt=gr.Textbox(label="Diagnosis / Clinical Notes",placeholder="Enter diagnosis or paste results...",lines=6)
                    with gr.Row():
                        tb=gr.Button("\U0001f48a Generate Plan",variant="primary",size="lg")
                        tc=gr.Button("\U0001f5d1\ufe0f Clear",variant="secondary")
                with gr.Column(scale=1):
                    to2=gr.HTML()
            tb.click(h_treatment,[tn,ta,tg,tbp,thr,ttp,tsp,th,tm,tnt],[to2])
            tc.click(lambda:"",outputs=[to2])

        # Tab 6: Summary
        with gr.TabItem("\U0001f4cb Clinical Summary"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f4cb Clinical Summary Generator")
                    gr.Markdown("Run any analysis first, then generate a structured case summary.")
                    sub=gr.Button("\U0001f4cb Generate Summary",variant="primary",size="lg")
                with gr.Column(scale=1):
                    suo=gr.HTML()
            sub.click(h_summary,[],[suo])

        # Tab 7: Patient Explanation
        with gr.TabItem("\U0001f464 Patient Explanation"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f464 Patient-Friendly Explanation")
                    pl=gr.Dropdown(LANGUAGES,label="Language",value="English")
                    pfi=gr.Textbox(label="Clinical Findings to Translate",placeholder="Paste findings or leave empty to use last analysis...",lines=8)
                    pb=gr.Button("\U0001f464 Explain to Patient",variant="primary",size="lg")
                with gr.Column(scale=1):
                    po=gr.HTML()
            pb.click(h_explain,[pfi,pl],[po])

        # Tab 8: Refine
        with gr.TabItem("\U0001f501 Refine & Feedback"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1):
                    gr.Markdown("### \U0001f501 Refinement Feedback")
                    gr.Markdown("Provide clinical feedback to refine the last analysis.")
                    ft=gr.Textbox(label="Doctor's Feedback",placeholder="e.g. 'Patient also has...', 'Consider ruling out...'",lines=5)
                    with gr.Row():
                        fb2=gr.Button("\U0001f501 Refine Analysis",variant="primary",size="lg")
                        fc=gr.Button("\U0001f5d1\ufe0f Clear All",variant="stop")
                with gr.Column(scale=1):
                    fo=gr.HTML()
            fb2.click(h_refine,[ft],[fo])
            fc.click(lambda:"",outputs=[fo])

    gr.HTML(FOOTER)

print("\u2705 App built! Ready to launch.")

/tmp/ipykernel_2683/2731627712.py:24: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="MediAI by Aigenthix") as app:
/tmp/ipykernel_2683/2731627712.py:24: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="MediAI by Aigenthix") as app:


✅ App built! Ready to launch.


## 🚀 Step 8: Launch

In [ ]:
app.launch(share=True, debug=True, show_error=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e9cd2dde0187b437c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🧪 Step 9: Test Examples (Optional)

In [ ]:
print("\U0001fa7a Example: Cardiology case...")
r=agent.run(patient_name="John Doe",age="55",gender="Male",blood_pressure="158/95",heart_rate="92",temperature="99.2\u00b0F",spo2="94%",medical_history="T2DM 10yrs, HTN, Ex-smoker",medications="Metformin, Amlodipine, Aspirin",symptoms="Chest tightness 3d, SOB on stairs, dizziness, L arm numbness",specialty="Cardiology",diagnosis_type="Differential Diagnosis",analysis_mode="symptom")
print(f"{get_risk_emoji(r['risk_score'])} {get_risk_label(r['risk_score'])} ({r['risk_score']}/100)")
print(r["final_output"][:400]+"...")

In [ ]:
print("\U0001f9ea Example: Lab Report...")
r=agent.run(patient_name="Sarah W",age="42",gender="Female",lab_values="Hb:10.2(Low), WBC:11800, MCV:72(Low), FG:118(High), HbA1c:6.2%, TSH:8.4(High), FreeT4:0.7(Low), Iron:35(Low), Ferritin:12(Low), Chol:242(High), LDL:158(High)",analysis_mode="lab")
print(f"{get_risk_emoji(r['risk_score'])} {get_risk_label(r['risk_score'])} ({r['risk_score']}/100)")
print(r["final_output"][:400]+"...")

---
### \u2705 Design System Match

| Element | React | Gradio |
|---|---|---|
| Header gradient | `#0F172A \u2192 #1E3A5F \u2192 #0F766E` | \u2705 Same |
| Font | DM Sans | \u2705 Same |
| Background | `#F0F4F8` | \u2705 Same |
| Card style | White, 16px radius, `#E2E8F0` border | \u2705 Same |
| Input style | `#F8FAFC` bg, 1.5px `#CBD5E1`, 10px radius | \u2705 Same |
| Primary button | Gradient `#0F766E \u2192 #0EA5E9`, 12px radius | \u2705 Same |
| Tab active | `#0F766E` color, 3px bottom border | \u2705 Same |
| Risk meter | Conic-gradient donut, 120px | \u2705 Same |
| KPI cards | Gradient bg with colored borders | \u2705 Same |
| Section blocks | Left 4px color border, `#F8FAFC` bg | \u2705 Same |
| Marquee | Animated horizontal scroll | \u2705 Same |
| Disclaimer | `#FFFBEB` bg, `#FDE68A` border | \u2705 Same |
| Badge | Green pill `v2.0` | \u2705 Same |
| AI status | Green dot with glow | \u2705 Same |

### Built by **Aigenthix** \U0001f3e5